## Explore and clean the data

Run from the project root or validation directory. This demonstration uses synthetic Data Analyst CV–posting pairs. The K-means occupation profiles are exploratory. This classifier uses the job requirement and the CV excerpt as its features.

In [1]:
from pathlib import Path
import sys
import pandas as pd

validation_dir = Path.cwd() if Path('run.py').exists() else Path.cwd() / 'validation'
sys.path.insert(0, str(validation_dir))
from run import (LABELS, baseline, build_pipeline, clean, evaluate,
                 features, make_demo, select_model, validate)

raw = make_demo()
print(raw.shape)
print(raw.dtypes)
print(raw.isna().mean().mul(100).round(2).rename('missing percent'))
print('Exact duplicates:', raw.duplicated().sum())
print(raw.label.value_counts())
data, cleaning = clean(raw)
print(cleaning)


(600, 8)
split               object
cv_id               object
posting_id          object
industry            object
role                object
requirement_text    object
cv_excerpt          object
label               object
dtype: object
split               0.0
cv_id               0.0
posting_id          0.0
industry            0.0
role                0.0
requirement_text    0.0
cv_excerpt          0.0
label               0.0
Name: missing percent, dtype: float64
Exact duplicates: 0
label
evidence found       204
unclear              198
no evidence found    198
Name: count, dtype: int64
{'input_rows': 600, 'missing_by_column': {'split': 0, 'cv_id': 0, 'posting_id': 0, 'industry': 0, 'role': 0, 'requirement_text': 0, 'cv_excerpt': 0, 'label': 0}, 'exact_duplicate_rows_removed': 0, 'rows_missing_required_values_removed': 0, 'optional_columns_dropped': []}


## Prepare features and check label balance

Use the job requirement and CV excerpt as X and the evidence label as y. Each synthetic CV–posting pair contributes four requirement examples. The generator gives each pair one example of each label and rotates the fourth label by pair index within each industry. A fixed seed shuffles the labels across requirements and selects wording within the split.

Training has 120 examples per label, so the pipeline fits the original training rows with equal weights. Validation and test have 42 found, 39 absent and 39 unclear examples each. These are the distributions generated for those splits. Macro F1 gives each label equal weight when comparing models.

In [2]:
validate(data)
train, validation, test = [data.loc[data.split.eq(split)]
                           for split in ('train', 'validation', 'test')]
X_train, y_train = features(train), train.label
X_val, y_val = features(validation), validation.label
X_test, y_test = features(test), test.label
print(data.groupby('split').agg(requirement_rows=('label', 'size'),
                                cvs=('cv_id', 'nunique'), postings=('posting_id', 'nunique')))
label_counts = pd.crosstab(data.split, data.label).reindex(
    index=['train', 'validation', 'test'], columns=LABELS
)
assert label_counts.loc['train'].nunique() == 1
print(label_counts.to_string())
print('Training share per label:', round(100 / len(LABELS), 1), '%')
print(X_train.shape, X_val.shape, X_test.shape)


            requirement_rows  cvs  postings
split                                      
test                     120   30        30
train                    360   90        90
validation               120   30        30
label       evidence found  no evidence found  unclear
split                                                 
train                  120                120      120
validation              42                 39       39
test                    42                 39       39
Training share per label: 33.3 %
(360, 2) (120, 2) (120, 2)


## Audit the split before fitting

Check each pair of splits for shared CV IDs, posting IDs and exact CV excerpts after ignoring case, punctuation and spacing. Job requirements such as SQL can recur. The excerpt check can catch copied text under different IDs. The synthetic splits use different sentence frames, and the check covers exact matches after normalisation.

Use only requirement and CV excerpt text as model inputs. Fit the TF-IDF vocabularies, IDF weights and classifiers on training data. Use validation macro F1 to select the model and setting, then score the frozen choice on test. For a real-data study, keep connected CVs and postings together, check near-duplicate text and use an independent test set with realistic label frequencies.

In [3]:
from itertools import combinations

partitions = {'Train': train, 'Validation': validation, 'Test': test}
normalised_excerpts = {
    name: set(part.cv_excerpt.str.casefold().str.replace(r'[^\w]+', ' ', regex=True).str.strip())
    for name, part in partitions.items()
}
overlap = pd.DataFrame([
    {
        'Pair': f'{left} / {right}',
        'Shared CV IDs': len(set(partitions[left].cv_id) & set(partitions[right].cv_id)),
        'Shared posting IDs': len(set(partitions[left].posting_id) & set(partitions[right].posting_id)),
        'Shared normalised CV excerpts': len(normalised_excerpts[left] & normalised_excerpts[right]),
    }
    for left, right in combinations(partitions, 2)
])
assert not overlap.drop(columns='Pair').to_numpy().any()
for features_in_split in (X_train, X_val, X_test):
    assert list(features_in_split.columns) == ['requirement_text', 'cv_excerpt']
print(overlap.to_string(index=False))
print('Model inputs:', ', '.join(X_train.columns))

              Pair  Shared CV IDs  Shared posting IDs  Shared normalised CV excerpts
Train / Validation              0                   0                              0
      Train / Test              0                   0                              0
 Validation / Test              0                   0                              0
Model inputs: requirement_text, cv_excerpt


## Build and tune the pipeline

Inspect the named preprocessing and classifier steps, like in Week 10. The two text columns use separate TF-IDF transformers. The selector uses set_params on each pipeline and ranks settings by validation macro F1.

Logistic regression provides an interpretable linear benchmark. Linear SVM tests a margin-based classifier on the same sparse features. Complement Naive Bayes tests a text model that uses the feature weights from competing classes. We choose the highest validation macro F1.

In [4]:
print(build_pipeline())
best_name, model, scores = select_model(X_train, y_train, X_val, y_val)
print({c: round(score['macro_f1'], 3) for c, score in scores.items()})
print('Selected model:', best_name)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('requirement',
                                                  TfidfVectorizer(ngram_range=(1,
                                                                               2)),
                                                  'requirement_text'),
                                                 ('cv',
                                                  TfidfVectorizer(ngram_range=(1,
                                                                               2)),
                                                  'cv_excerpt')])),
                ('classifier', LogisticRegression(max_iter=1000))])


{'Logistic regression (C=0.1)': 0.193, 'Logistic regression (C=1.0)': 0.232, 'Logistic regression (C=10.0)': 0.232, 'Linear SVM (C=0.1)': 0.288, 'Linear SVM (C=1.0)': 0.278, 'Linear SVM (C=10.0)': 0.278, 'Complement naive Bayes (alpha=0.1)': 0.451, 'Complement naive Bayes (alpha=1.0)': 0.437, 'Complement naive Bayes (alpha=10.0)': 0.432}
Selected model: Complement naive Bayes (alpha=0.1)


## Evaluate on the test split

The model is evaluated after selecting a setting. These scores describe synthetic examples. Examine the false evidence matches before considering any real use.

The majority baseline predicts one training label throughout. All three labels tie in this benchmark, so it uses the alphabetically first label. The phrase baseline checks whether the normalised job skill is present in the CV excerpt. The baselines and selected model are compared on the same held-out test rows.

In [5]:
from sklearn.metrics import classification_report

prediction = model.predict(X_test)
result = evaluate(y_test, prediction)
print(classification_report(y_test, prediction, labels=LABELS, zero_division=0))
print('Model macro F1:', round(result['macro_f1'], 3))
print('Keyword baseline macro F1:', round(evaluate(y_test, baseline(test))['macro_f1'], 3))
majority_label = y_train.mode().iloc[0]
print('Majority baseline macro F1:', round(evaluate(y_test, [majority_label] * len(test))['macro_f1'], 3))
matrix = pd.DataFrame(result['confusion_matrix'], index=LABELS, columns=LABELS)
print(matrix.rename_axis(index='actual', columns='predicted').to_string())


                   precision    recall  f1-score   support

   evidence found       0.54      1.00      0.70        42
no evidence found       0.89      0.62      0.73        39
          unclear       0.93      0.36      0.52        39

         accuracy                           0.67       120
        macro avg       0.79      0.66      0.65       120
     weighted avg       0.78      0.67      0.65       120

Model macro F1: 0.649
Keyword baseline macro F1: 0.289
Majority baseline macro F1: 0.173
predicted          evidence found  no evidence found  unclear
actual                                                       
evidence found                 42                  0        0
no evidence found              14                 24        1
unclear                        22                  3       14


### Where the classifier struggles

The classifier identified 14 of 39 unclear examples, giving that class a recall of 35.9%. It also flagged 36 of 78 absent or unclear examples as evidence found. Equal numbers in training give each label equal representation, while the per-label results show how often the model recognises it.